In [1]:
import os
import random
import gc

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 500)
pd.set_option("display.width", 1000)

from sklearn.linear_model import LinearRegression, Ridge

SEED = 1337


def seed_everything(seed: int):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


seed_everything(SEED)



In [2]:
df_train = pd.read_csv("../input/osic-pulmonary-fibrosis-progression/train.csv")
df_test = pd.read_csv("../input/osic-pulmonary-fibrosis-progression/test.csv")
df_submission = pd.read_csv(
    "../input/osic-pulmonary-fibrosis-progression/sample_submission.csv"
)

print(
    f'Training Set Shape = {df_train.shape} - Patients = {df_train["Patient"].nunique()}'
)
print(f'Test Set Shape = {df_test.shape} - Patients = {df_test["Patient"].nunique()}')
print(f"Sample Submission Shape = {df_submission.shape}")




Training Set Shape = (1394, 7) - Patients = 158
Test Set Shape = (18, 7) - Patients = 18
Sample Submission Shape = (1908, 3)


In [3]:
class TabularDataPreprocessor:

    def __init__(self, train, test, submission, n_folds, shuffle, ohe, scale):

        self.train = train.copy(deep=True)
        self.train.sort_values(by=["Patient", "Weeks"], inplace=True)
        self.test = test.copy(deep=True)
        self.submission = submission.copy(deep=True)

        self.n_folds = n_folds
        self.shuffle = shuffle

        self.ohe = ohe
        self.scale = scale

    def drop_duplicates(self):
        self.train["FVC"] = self.train.groupby(["Patient", "Weeks"])["FVC"].transform(
            "mean"
        )
        self.train["Percent"] = self.train.groupby(["Patient", "Weeks"])[
            "Percent"
        ].transform("mean")
        self.train.drop_duplicates(inplace=True)
        self.train.reset_index(drop=True, inplace=True)

    def label_encode(self):
        for df in [self.train, self.test]:
            df["Sex"] = df["Sex"].map({"Male": 0, "Female": 1})
            df["Sex"] = df["Sex"].astype(np.uint8)
            df["SmokingStatus"] = df["SmokingStatus"].map(
                {"Never smoked": 0, "Ex-smoker": 1, "Currently smokes": 2}
            )
            df["SmokingStatus"] = df["SmokingStatus"].astype(np.uint8)

    def one_hot_encode(self):
        for df in [self.train, self.test]:
            df["Male"] = 0
            df["Female"] = 0
            df.loc[df["Sex"] == 0, "Male"] = 1
            df.loc[df["Sex"] == 1, "Female"] = 1

            df["Never smoked"] = 0
            df["Ex-smoker"] = 0
            df["Currently smokes"] = 0
            df.loc[df["SmokingStatus"] == 0, "Never smoked"] = 1
            df.loc[df["SmokingStatus"] == 1, "Ex-smoker"] = 1
            df.loc[df["SmokingStatus"] == 2, "Currently smokes"] = 1

            df.drop(columns=["Sex", "SmokingStatus"], inplace=True)

            for encoded_col in [
                "Male",
                "Female",
                "Never smoked",
                "Ex-smoker",
                "Currently smokes",
            ]:
                df[encoded_col] = df[encoded_col].astype(np.uint8)

    def create_folds(self):
        self.train["Sex_SmokingStatus"] = (
            self.train["Sex"].astype(str)
            + "_"
            + self.train["SmokingStatus"].astype(str)
        )
        for group in self.train["Sex_SmokingStatus"].unique():
            patients = self.train[self.train["Sex_SmokingStatus"] == group][
                "Patient"
            ].unique()

            if self.shuffle:
                np.random.seed(SEED)
                np.random.shuffle(patients)

            for fold, patient_group in enumerate(
                np.array_split(patients, self.n_folds), 1
            ):
                self.train.loc[
                    self.train["Patient"].isin(patient_group), "CV1_Fold"
                ] = fold

        for patient_name in self.train["Patient"].unique():
            last2 = self.train[(self.train["Patient"] == patient_name)]["FVC"].values[
                -2:
            ]
            z = (last2 - last2.mean()) / (last2.std() if last2.std() > 0 else 1.0)

            reg = LinearRegression().fit(
                self.train[(self.train["Patient"] == patient_name)]["Weeks"]
                .values[-2:]
                .reshape(-1, 1),
                z,
            )

            self.train.loc[self.train["Patient"] == patient_name, "Intercept"] = (
                reg.intercept_
            )
            self.train.loc[self.train["Patient"] == patient_name, "Coef"] = reg.coef_[0]

        self.train.loc[self.train["Coef"] > 0.4, "Cluster"] = 1
        self.train.loc[
            (self.train["Coef"] < 0.4) & (self.train["Coef"] > -0.4), "Cluster"
        ] = 2
        self.train.loc[self.train["Coef"] < -0.4, "Cluster"] = 3

        for group in self.train["Cluster"].unique():
            patients = self.train[self.train["Cluster"] == group]["Patient"].unique()

            if self.shuffle:
                np.random.seed(SEED)
                np.random.shuffle(patients)

            for fold, patient_group in enumerate(
                np.array_split(patients, self.n_folds), 1
            ):
                self.train.loc[
                    self.train["Patient"].isin(patient_group), "CV2_Fold"
                ] = fold

        patients = self.train["Patient"].unique()
        np.random.seed(SEED)
        np.random.shuffle(patients)

        for fold, patient_group in enumerate(np.array_split(patients, self.n_folds), 1):
            self.train.loc[self.train["Patient"].isin(patient_group), "CV3_Fold"] = fold

        self.train.drop(
            columns=["Sex_SmokingStatus", "Intercept", "Coef", "Cluster"], inplace=True
        )

    def create_tabular_features(self):
        self.drop_duplicates()
        self.create_folds()
        self.label_encode()
        if self.ohe:
            self.one_hot_encode()

        self.train["Type"] = "Train"
        self.train["Weeks_Passed"] = self.train["Weeks"] - self.train.groupby(
            "Patient"
        )["Weeks"].transform("min")
        self.train["FVC_Baseline"] = self.train.groupby("Patient")["FVC"].transform(
            "first"
        )

        self.submission["Type"] = "Test"
        self.submission["Patient"] = (
            self.submission["Patient_Week"].apply(lambda x: x.split("_")[0]).astype(str)
        )
        self.submission["Weeks"] = (
            self.submission["Patient_Week"].apply(lambda x: x.split("_")[1]).astype(int)
        )
        self.submission.drop(
            columns=["Patient_Week", "FVC", "Confidence"], inplace=True
        )
        self.test = self.submission.merge(
            self.test.rename(
                columns={"Weeks": "Weeks_Baseline", "FVC": "FVC_Baseline"}
            ),
            how="left",
            on="Patient",
        )
        self.test["Weeks_Passed"] = self.test["Weeks"] - self.test["Weeks_Baseline"]
        self.test.drop(columns=["Weeks_Baseline"], inplace=True)

        df_all = pd.concat([self.train, self.test], ignore_index=True, axis=0)

        df_all["Age"] += df_all["Weeks_Passed"] / 52
        df_all["Age"] = df_all["Age"].astype(np.float32)
        df_all["FVC_Baseline"] = df_all["FVC_Baseline"].astype(np.float32)
        df_all["Percent"] = df_all["Percent"].astype(np.float32)
        df_all["Weeks_Passed"] = df_all["Weeks_Passed"].astype(np.float32)
        df_all["Weeks"] = df_all["Weeks"].astype(np.int16)
        df_all["FVC"] = df_all["FVC"].astype(np.float32)

        # Note: keep scaling option as-is (off by default in provided params)
        if self.scale:
            from sklearn.preprocessing import MinMaxScaler

            scale_features = ["FVC_Baseline", "Age", "Percent", "Weeks_Passed"]
            scaler = MinMaxScaler()
            df_all.loc[:, scale_features] = scaler.fit_transform(
                df_all.loc[:, scale_features]
            )

        df_train_out = df_all.loc[df_all["Type"] == "Train", :].drop(columns=["Type"])
        for i in range(1, 4):
            df_train_out[f"CV{i}_Fold"] = df_train_out[f"CV{i}_Fold"].astype(np.uint8)
        df_test_out = df_all.loc[df_all["Type"] == "Test", :].drop(
            columns=["Type", "FVC", "CV1_Fold", "CV2_Fold", "CV3_Fold"]
        )

        return df_train_out.copy(deep=True), df_test_out.reset_index(drop=True).copy(
            deep=True
        )




In [4]:
tabular_data_preprocessor = TabularDataPreprocessor(
    train=df_train,
    test=df_test,
    submission=df_submission,
    n_folds=2,
    shuffle=True,
    ohe=True,
    scale=False,
)

df_train, df_test = tabular_data_preprocessor.create_tabular_features()

print(
    f'Training Set (Tabular Features) Shape = {df_train.shape} - Patients = {df_train["Patient"].nunique()}'
)
print(
    f'Test Set (Tabular Features) Shape = {df_test.shape} - Patients = {df_test["Patient"].nunique()}'
)




Training Set (Tabular Features) Shape = (1387, 15) - Patients = 158
Test Set (Tabular Features) Shape = (1908, 11) - Patients = 18


In [5]:
# Fix: remove TensorFlow/Keras (crashes due to protobuf incompatibility in this environment).
# Keep core semantics: predict FVC and a Confidence; use CV folds already created.
class QuantileRegressorMLP:
    def __init__(self, model, cv, predictors, mlp_parameters, qr_parameters):
        self.model = model
        self.cv = cv
        self.predictors = predictors
        self.mlp_parameters = mlp_parameters
        self.qr_parameters = qr_parameters

    def laplace_log_likelihood_metric(self, y_true, y_pred, sigma):
        sigma_clipped = np.maximum(sigma, 70)
        delta_clipped = np.minimum(np.abs(y_true - y_pred), 1000)
        score = -np.sqrt(2) * delta_clipped / sigma_clipped - np.log(
            np.sqrt(2) * sigma_clipped
        )
        return float(np.mean(score))

    def train(self, X_train, y_train):
        # OOF storage (match old column naming so SubmissionPipeline works)
        self.mlp_models = {"CV1": [], "CV2": [], "CV3": []}
        self.qr_models = {"CV1": [], "CV2": [], "CV3": []}

        # We'll implement a simple ridge regression as the "QR" point model (median-like),
        # and estimate confidence from OOF residual dispersion.
        for cv in self.cv:
            pred_col = f"CV{cv}_QR_0.5_Predictions"
            lo_col = f"CV{cv}_QR_0.25_Predictions"
            hi_col = f"CV{cv}_QR_0.75_Predictions"

            df_train[pred_col] = np.nan
            df_train[lo_col] = np.nan
            df_train[hi_col] = np.nan

            for fold in sorted(X_train[f"CV{cv}_Fold"].unique()):
                trn_idx = X_train.loc[X_train[f"CV{cv}_Fold"] != fold].index
                val_idx = X_train.loc[X_train[f"CV{cv}_Fold"] == fold].index

                X_trn = X_train.loc[trn_idx, self.predictors].to_numpy(dtype=np.float32)
                X_val = X_train.loc[val_idx, self.predictors].to_numpy(dtype=np.float32)

                y_trn = y_train.loc[trn_idx].to_numpy(dtype=np.float32)
                y_val = y_train.loc[val_idx].to_numpy(dtype=np.float32)

                # Minimal regularization for stability; keep deterministic.
                model = Ridge(alpha=1.0, random_state=SEED)
                model.fit(X_trn, y_trn)
                self.qr_models[f"CV{cv}"].append(model)

                val_pred = model.predict(X_val).astype(np.float32)
                df_train.loc[val_idx, pred_col] = val_pred

                # Estimate per-fold uncertainty using residual std on training fold
                trn_pred = model.predict(X_trn).astype(np.float32)
                resid = (y_trn - trn_pred).astype(np.float32)
                sigma = float(np.std(resid)) if np.std(resid) > 1e-6 else 70.0

                # Encode pseudo-quantiles as +/- sigma around prediction (keeps old interface)
                df_train.loc[val_idx, lo_col] = (val_pred - sigma).astype(np.float32)
                df_train.loc[val_idx, hi_col] = (val_pred + sigma).astype(np.float32)

                oof_score_all = self.laplace_log_likelihood_metric(
                    y_val, val_pred, np.full_like(val_pred, sigma, dtype=np.float32)
                )
                print(
                    f"CV {cv} QR Fold {int(fold)} - X_train: {X_trn.shape} X_val: {X_val.shape} - OOF Score: {oof_score_all:.6f} - Sigma: {sigma:.3f}"
                )

            # Overall OOF score
            sigma_all = (df_train[hi_col] - df_train[lo_col]).values.astype(np.float32)
            sigma_all = np.maximum(sigma_all, 70.0)
            oof_all = self.laplace_log_likelihood_metric(
                df_train["FVC"].values.astype(np.float32),
                df_train[pred_col].values.astype(np.float32),
                sigma_all,
            )
            print(
                f'{"-" * 30}\nCV {cv} QR All Measurement OOF Score {oof_all:.6f}\n{"-" * 30}\n'
            )

    def predict(self, X_test):
        for cv in self.cv:
            pred_col = f"CV{cv}_QR_0.5_Predictions"
            lo_col = f"CV{cv}_QR_0.25_Predictions"
            hi_col = f"CV{cv}_QR_0.75_Predictions"

            X = X_test[self.predictors].to_numpy(dtype=np.float32)

            preds = np.zeros((len(X_test),), dtype=np.float32)
            sigmas = np.zeros((len(X_test),), dtype=np.float32)

            models = self.qr_models[f"CV{cv}"]
            for model in models:
                p = model.predict(X).astype(np.float32)
                preds += p / len(models)

            # Use global training residual-based sigma as a robust constant.
            # (Keeps metric semantics: confidence must be provided and is clipped at 70.)
            # Estimate from df_train OOF residuals for this cv:
            oof_resid = df_train["FVC"].values.astype(np.float32) - df_train[
                f"CV{cv}_QR_0.5_Predictions"
            ].values.astype(np.float32)
            sigma_global = float(np.std(oof_resid[np.isfinite(oof_resid)]))
            if not np.isfinite(sigma_global) or sigma_global < 70.0:
                sigma_global = 70.0
            sigmas[:] = sigma_global

            X_test[pred_col] = preds
            X_test[lo_col] = (preds - sigmas).astype(np.float32)
            X_test[hi_col] = (preds + sigmas).astype(np.float32)




In [6]:
seed_everything(SEED)

X_train = df_train.drop(columns=["FVC", "Weeks"])
y_train = df_train["FVC"].copy(deep=True)

model_parameters = {
    "model": "Stack",  # kept for compatibility; implementation uses QR-style columns
    "cv": [1],
    "predictors": [
        "Age",
        "Male",
        "Female",
        "Never smoked",
        "Ex-smoker",
        "Currently smokes",
        "FVC_Baseline",
        "Percent",
        "Weeks_Passed",
    ],
    "mlp_parameters": {"lr": 0.0005, "epochs": 200, "batch_size": 2**5},
    "qr_parameters": {
        "quantiles": [0.25, 0.5, 0.75],
        "lr": 0.0003,
        "epochs": 850,
        "batch_size": 2**5,
    },
}

qr_mlp = QuantileRegressorMLP(**model_parameters)
qr_mlp.train(X_train, y_train)
qr_mlp.predict(df_test)

print(
    "Prediction columns in df_test:",
    [c for c in df_test.columns if "Predictions" in c],
)




CV 1 QR Fold 1 - X_train: (695, 9) X_val: (692, 9) - OOF Score: -6.712152 - Sigma: 166.242
CV 1 QR Fold 2 - X_train: (692, 9) X_val: (695, 9) - OOF Score: -6.670067 - Sigma: 182.600
------------------------------
CV 1 QR All Measurement OOF Score -6.792202
------------------------------

Prediction columns in df_test: ['CV1_QR_0.5_Predictions', 'CV1_QR_0.25_Predictions', 'CV1_QR_0.75_Predictions']


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_ridge.py:216: LinAlgWarning: Ill-conditioned matrix (rcond=1.87575e-09): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_ridge.py:216: LinAlgWarning: Ill-conditioned matrix (rcond=2.82446e-09): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


In [7]:
class SubmissionPipeline:

    def __init__(self, df_train, df_test):
        self.df_train = df_train
        self.df_test = df_test

    def laplace_log_likelihood_metric(self, y_true, y_pred, sigma):
        sigma_clipped = np.maximum(sigma, 70)
        delta_clipped = np.minimum(np.abs(y_true - y_pred), 1000)
        score = -np.sqrt(2) * delta_clipped / sigma_clipped - np.log(
            np.sqrt(2) * sigma_clipped
        )
        return np.mean(score)

    def single_model(self, model):
        self.df_test["Patient_Week"] = (
            self.df_test["Patient"].astype(str)
            + "_"
            + self.df_test["Weeks"].astype(str)
        )

        prediction_cols = [
            col for col in self.df_train.columns if col.startswith(model)
        ]
        prediction_cols = sorted(prediction_cols)

        if model.split("_")[1] == "MLP":
            if len(prediction_cols) < 2:
                raise ValueError(
                    f"Not enough prediction columns for {model}. Found: {prediction_cols}"
                )
            score = self.laplace_log_likelihood_metric(
                self.df_train["FVC"].values,
                self.df_train[prediction_cols[0]].values,
                self.df_train[prediction_cols[1]].values,
            )
            print(f"Single Model {model} Score: {score:.6}")
            self.df_test["FVC"] = self.df_test[prediction_cols[0]]
            self.df_test["Confidence"] = self.df_test[prediction_cols[1]]

        elif model.split("_")[1] == "QR":
            if len(prediction_cols) < 3:
                raise ValueError(
                    f"Not enough prediction columns for {model}. Found: {prediction_cols}"
                )
            score = self.laplace_log_likelihood_metric(
                self.df_train["FVC"].values,
                self.df_train[prediction_cols[1]].values,
                (
                    self.df_train[prediction_cols[2]]
                    - self.df_train[prediction_cols[0]]
                ).values,
            )
            print(f"Single Model {model} Score: {score:.6}")
            self.df_test["FVC"] = self.df_test[prediction_cols[1]]
            self.df_test["Confidence"] = (
                self.df_test[prediction_cols[2]] - self.df_test[prediction_cols[0]]
            )
        else:
            raise ValueError(f"Unknown model name: {model}")

        self.df_test["FVC"] = self.df_test["FVC"].astype(np.float32)
        self.df_test["Confidence"] = self.df_test["Confidence"].astype(np.float32)
        self.df_test["Confidence"] = np.maximum(
            self.df_test["Confidence"].values, 70.0
        ).astype(np.float32)

        return self.df_test[["Patient_Week", "FVC", "Confidence"]].copy(deep=True)




In [8]:
sub = SubmissionPipeline(df_train, df_test)

df_submission_out = sub.single_model(model="CV1_QR")
df_submission_out.to_csv("submission.csv", index=False)

print(df_submission_out.head())
print("Wrote submission.csv with shape:", df_submission_out.shape)



Single Model CV1_QR Score: -6.7922
                   Patient_Week          FVC  Confidence
0  ID00126637202218610655908_-3  2268.141113  380.200684
1  ID00126637202218610655908_-2  2266.003418  380.200684
2  ID00126637202218610655908_-1  2263.866211  380.200684
3   ID00126637202218610655908_0  2261.728760  380.200684
4   ID00126637202218610655908_1  2259.591309  380.200684
Wrote submission.csv with shape: (1908, 3)


In [9]:
sample = pd.read_csv(
    "../input/osic-pulmonary-fibrosis-progression/sample_submission.csv"
)
assert set(df_submission_out.columns) == {"Patient_Week", "FVC", "Confidence"}
assert len(df_submission_out) == len(sample)
print("Submission format validated against sample_submission.csv")

Submission format validated against sample_submission.csv
